In [1]:
# ── Mount Drive + verify path ─────────────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")

# Find the exact folder name (screenshot shows it's truncated)
base = "/content/drive/MyDrive"
for folder in os.listdir(base):
    if "ACL" in folder or "Sign" in folder:
        print("Found:", folder)

# Then list the checkpoints folder
CHECKPOINT_PATH = "/content/drive/MyDrive/ACL_SignLanguage_Research/checkpoints/TransNET.pth"
print("\nFile exists:", os.path.exists(CHECKPOINT_PATH))

# If False, find the right path:
if not os.path.exists(CHECKPOINT_PATH):
    for root, dirs, files in os.walk(base):
        for f in files:
            if "TransNET" in f:
                print("Found at:", os.path.join(root, f))

Mounted at /content/drive
Found: ACL_SignLanguage_Research

File exists: True


In [4]:
# ─── CASL-TransNet inference on KSL-W30 test set ─────────────────────────
import torch, torch.nn as nn, math, numpy as np, pandas as pd
from datasets import load_dataset

# ── TransSLR architecture (must match CASL training) ─────────────────────
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TransSLR(nn.Module):
    def __init__(self, feat_dim, d_model, nhead, num_layers, ffn_dim, dropout, num_classes):
        super().__init__()
        self.proj    = nn.Linear(feat_dim, d_model)
        self.pos_enc = SinusoidalPE(d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True, norm_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.drop    = nn.Dropout(dropout)
        self.head    = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.proj(x); x = self.pos_enc(x); x = self.encoder(x)
        x = self.gap(x.transpose(1, 2)).squeeze(-1)
        return self.head(self.drop(x))

In [6]:
import torch, torch.nn as nn, math, numpy as np, pandas as pd
from datasets import load_dataset

# ── Correct TransNET architecture (reverse-engineered from checkpoint) ────
class TransNET(nn.Module):
    def __init__(self, input_dim=225, d_model=512, nhead=8,
                 num_layers=4, ffn_dim=1024, num_classes=60, seq_len=64):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, d_model),   # (B, 64, 225) → (B, 64, 512)
            nn.BatchNorm1d(seq_len),          # treats seq_len as channels
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),   # .0
            nn.ReLU(),                 # .1
            nn.Dropout(0.5),           # .2
            nn.Linear(256, num_classes),  # .3
        )

    def forward(self, x):              # x: (B, T, 225)
        x = self.feature_extractor(x)  # (B, T, 512)
        x = self.transformer(x)        # (B, T, 512)
        x = x.mean(dim=1)             # (B, 512) — global average pool
        return self.classifier(x)      # (B, 60)

# ── Load checkpoint ───────────────────────────────────────────────────────
CHECKPOINT_PATH = "/content/drive/MyDrive/ACL_SignLanguage_Research/checkpoints/TransNET.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt       = torch.load(CHECKPOINT_PATH, map_location=device)
state_dict = ckpt["model_state_dict"]

num_classes_model = state_dict["classifier.3.weight"].shape[0]   # 60
d_model           = state_dict["classifier.0.weight"].shape[1]   # 512
print(f"✅ Detected: d_model={d_model}, num_classes={num_classes_model}, val_acc={ckpt['val_acc']:.4f}")

model = TransNET(
    input_dim=225, d_model=d_model, nhead=8,
    num_layers=4, ffn_dim=1024,
    num_classes=num_classes_model, seq_len=64
).to(device)
model.load_state_dict(state_dict)
model.eval()
print(f"✅ TransNET loaded — {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

# ── Build CASL label vocab (maps index → sign name) ───────────────────────
print("Loading CASL label vocab...")
casl_ds   = load_dataset("luciayen/CASL-W60-Landmarks", split="train")
casl_labels = sorted(set(casl_ds["label"]))
assert len(casl_labels) == num_classes_model, \
    f"Expected {num_classes_model} CASL labels, got {len(casl_labels)}"
idx2label_casl = {i: l for i, l in enumerate(casl_labels)}
print(f"✅ CASL vocab: {casl_labels}")

# ── Load KSL test set ─────────────────────────────────────────────────────
print("Loading KSL-W30 test set...")
ksl_test = load_dataset("luciayen/KSL-W30-Landmarks", split="test")
print(f"✅ KSL test samples: {len(ksl_test)}")

# ── Run inference ─────────────────────────────────────────────────────────
rows = []
with torch.no_grad():
    for i, sample in enumerate(ksl_test):
        lm     = torch.tensor(sample["landmarks"], dtype=torch.float32).unsqueeze(0).to(device)
        logits = model(lm)
        probs  = torch.softmax(logits, dim=1)
        pred_idx   = logits.argmax(dim=1).item()
        confidence = probs[0, pred_idx].item()
        pred_label = idx2label_casl[pred_idx]
        actual     = sample["label"]
        correct    = pred_label.lower() == actual.lower()
        rows.append({
            "Video ID":   f"test_{i:04d}",
            "Actual":     actual,
            "Predicted":  pred_label,
            "Confidence": f"{confidence*100:.1f}%",
            "Result":     "✅" if correct else "❌",
        })

# ── Results table ─────────────────────────────────────────────────────────
df  = pd.DataFrame(rows)
acc = (df["Result"] == "✅").mean()

print(f"\n{'='*60}")
print(f"  CASL-TransNET → KSL-W30 Transfer Evaluation")
print(f"{'='*60}")
print(f"  Accuracy : {acc*100:.2f}%  ({(df['Result']=='✅').sum()}/{len(df)})")
print(f"  Note: CASL (60 classes) predicting KSL (30 classes)")
print(f"  'Correct' = predicted CASL word matches KSL word exactly")
print(f"{'='*60}\n")

# Per-class breakdown
per_class = (df.groupby("Actual")
               .apply(lambda g: pd.Series({
                   "Total":   len(g),
                   "Correct": (g["Result"]=="✅").sum(),
                   "Acc%":    f"{(g['Result']=='✅').mean()*100:.0f}%",
                   "Top Prediction": g["Predicted"].mode()[0],
               }))
               .reset_index())
print("Per-class summary:")
print(per_class.to_string(index=False))

print("\nFull prediction table:")
pd.set_option("display.max_rows", None)
display(df)

✅ Detected: d_model=512, num_classes=60, val_acc=80.3944
✅ TransNET loaded — 8.67M params
Loading CASL label vocab...


README.md:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/165M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3667 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ KSL test samples: 450

  CASL-TransNET → KSL-W30 Transfer Evaluation
  Accuracy : 0.00%  (0/450)
  Note: CASL (60 classes) predicting KSL (30 classes)
  'Correct' = predicted CASL word matches KSL word exactly

Per-class summary:
   Actual  Total  Correct Acc% Top Prediction
Agreement     15        0   0%           G003
    Apple     15        0   0%           G003
   Colour     15        0   0%           G004
   Friend     15        0   0%           G004
     Gift     15        0   0%           G003
   Market     15        0   0%           G003
   Monday     15        0   0%           G004
   No_100     15        0   0%           G004
   No_125     15        0   0%           G004
    No_17     15        0   0%           G004
    No_22     15        0   0%           G004
   No_268     15        0   0%           G004
    No_35     15        0   0%           G004
   No_388     15        0   0%           G004
   No_444     15        0   0%           G004
    No_48     15        0   0%  

/tmp/ipykernel_545/1093804148.py:100: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,Video ID,Actual,Predicted,Confidence,Result
0,test_0000,Agreement,G003,15.9%,❌
1,test_0001,Agreement,G004,23.9%,❌
2,test_0002,Agreement,G004,21.0%,❌
3,test_0003,Agreement,G003,22.9%,❌
4,test_0004,Agreement,G004,19.3%,❌
5,test_0005,Agreement,G054,18.2%,❌
6,test_0006,Agreement,G054,15.6%,❌
7,test_0007,Agreement,G003,19.6%,❌
8,test_0008,Agreement,G004,20.5%,❌
9,test_0009,Agreement,G003,32.1%,❌


In [7]:
# See which CASL words the model predicts most for KSL data
print("Most common CASL predictions on KSL test set:")
print(df["Predicted"].value_counts().head(15))

print("\nAny overlap between CASL and KSL vocabularies?")
ksl_classes  = set(ksl_test["label"])
casl_classes = set(casl_labels)
overlap      = ksl_classes & casl_classes
print(f"Shared class names: {overlap if overlap else 'NONE'}")

Most common CASL predictions on KSL test set:
Predicted
G004    195
G003    148
G054     99
G027      8
Name: count, dtype: int64

Any overlap between CASL and KSL vocabularies?
Shared class names: NONE
